In [2]:
!pip install -q transformers datasets

In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

# Veriyi yükle
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"Train: {train.shape}, Test: {test.shape}")

# BERT için MİNİMAL temizlik - sadece URL'leri sil, gerisine dokunma
def light_clean(text):
    text = str(text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["bert_text"] = train["text"].apply(light_clean)
test["bert_text"] = test["text"].apply(light_clean)

# Aynı stratejiyi koruyoruz: shuffle'lı, stratified split
X_train, X_val, y_train, y_val = train_test_split(
    train["bert_text"], train["target"],
    test_size=0.2, random_state=42, stratify=train["target"]
)

print(f"\nTrain örneği: {X_train.iloc[0]}")
print(f"Train boyutu: {len(X_train)}, Val boyutu: {len(X_val)}")

Train: (7613, 5), Test: (3263, 4)

Train örneği: Sassy city girl country hunk stranded in Smoky Mountain snowstorm #AoMS #ibooklove #bookboost
Train boyutu: 6090, Val boyutu: 1523


In [4]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Bir örnek üzerinde nasıl çalıştığını görelim
sample_text = X_train.iloc[0]
tokens = tokenizer(sample_text, truncation=True, max_length=64, padding="max_length")

print("Orijinal metin:", sample_text)
print("\nToken ID'leri (ilk 20):", tokens["input_ids"][:20])
print("\nAttention mask (ilk 20):", tokens["attention_mask"][:20])
print("\nGeri çözümlenmiş (decode edilmiş) hali:")
print(tokenizer.decode(tokens["input_ids"]))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Orijinal metin: Sassy city girl country hunk stranded in Smoky Mountain snowstorm #AoMS #ibooklove #bookboost

Token ID'leri (ilk 20): [101, 21871, 6508, 2103, 2611, 2406, 15876, 8950, 15577, 1999, 20629, 3137, 4586, 19718, 1001, 20118, 5244, 1001, 21307, 14659]

Attention mask (ilk 20): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Geri çözümlenmiş (decode edilmiş) hali:
[CLS] sassy city girl country hunk stranded in smoky mountain snowstorm # aoms # ibooklove # bookboost [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


In [5]:
from datasets import Dataset

# pandas -> HuggingFace Dataset formatına çevir
train_ds = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
val_ds = Dataset.from_dict({"text": X_val.tolist(), "label": y_val.tolist()})

# Tokenize eden fonksiyon (batch halinde çalışır, hızlıdır)
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=64, padding="max_length")

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)

print(train_ds)
print("\nÖrnek bir satır:", train_ds[0].keys())

Map:   0%|          | 0/6090 [00:00<?, ? examples/s]

Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 6090
})

Örnek bir satır: dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2   # binary classification: afet mi değil mi
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)   # en yüksek olasılıklı sınıfı seç
    f1 = f1_score(labels, predictions)
    acc = accuracy_score(labels, predictions)
    return {"f1": f1, "accuracy": acc}

In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_output",
    eval_strategy="epoch",       # her epoch sonunda validation'da ölç
    save_strategy="epoch",       # her epoch sonunda modeli kaydet
    learning_rate=2e-5,          # BERT fine-tuning için tipik, küçük bir öğrenme oranı
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,          # az başlıyoruz, sonra artırabiliriz
    weight_decay=0.01,           # overfitting'e karşı hafif bir düzenlileştirme
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",            # wandb gibi harici loglama servislerini kapat
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [9]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.445374,0.376931,0.815119,0.852265
2,0.336164,0.383854,0.816521,0.848326


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=762, training_loss=0.38538604458486003, metrics={'train_runtime': 101.7842, 'train_samples_per_second': 119.665, 'train_steps_per_second': 7.486, 'total_flos': 201681614453760.0, 'train_loss': 0.38538604458486003, 'epoch': 2.0})

In [10]:
# Test verisini de aynı şekilde tokenize edip Dataset'e çevir
test_ds = Dataset.from_dict({"text": test["bert_text"].tolist()})
test_ds = test_ds.map(tokenize_fn, batched=True)

# Tahmin yap
predictions = trainer.predict(test_ds)
test_preds = np.argmax(predictions.predictions, axis=1)

print("Tahmin dağılımı:", np.bincount(test_preds))
print("İlk 10 tahmin:", test_preds[:10])

Map:   0%|          | 0/3263 [00:00<?, ? examples/s]

Tahmin dağılımı: [2019 1244]
İlk 10 tahmin: [1 1 1 1 1 1 0 0 0 0]


In [11]:
submission = pd.DataFrame({
    "id": test["id"],
    "target": test_preds
})
submission.to_csv("submission_distilbert.csv", index=False)
print(submission.head())

# Colab'dan bilgisayarına indir
from google.colab import files
files.download("submission_distilbert.csv")

   id  target
0   0       1
1   2       1
2   3       1
3   9       1
4  11       1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>